# Blinkit Grocery Sales Analysis
## 02 – Data Quality Assessment and Cleaning
### One command / operation per Jupyter code cell

**Input:** `Dataset/BlinkIT-Grocery-Data.csv`  
**Output:** `Dataset/blinkit_sales_cleaned.csv`

**Objective:** Resolve identified data-quality issues while preserving the original source file.

### Cleaning decisions
1. Standardize `Item Fat Content`: `LF` and `low fat` become `Low Fat`; `reg` becomes `Regular`.
2. Impute missing `Item Weight` in stages: same `Item Identifier` median, then same `Item Type` median, then overall median.
3. Retain zero `Item Visibility` values until their meaning is confirmed.
4. Detect potential numerical outliers using IQR, but do not delete them automatically.
5. Remove exact duplicate rows if any are present.
6. Validate the result and save a separate cleaned CSV.

> Run from the `Notebooks` folder. Each command or small operation has its own code cell.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

Set Pandas display options.

In [93]:
pd.set_option('display.max_columns', None)

Set a readable floating-point display format.

In [94]:
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

## 2. Load the raw dataset

Set the raw dataset path.

In [96]:
possible_paths = [
    Path("../Dataset/BlinkIT-Grocery-Data.csv"),
    Path("Dataset/BlinkIT-Grocery-Data.csv"),
    Path("BlinkIT-Grocery-Data.csv"),
    Path("../BlinkIT-Grocery-Data.csv"),
]

INPUT_PATH = next((path for path in possible_paths if path.exists()), None)

if INPUT_PATH is None:
    raise FileNotFoundError(
        "Could not find BlinkIT-Grocery-Data.csv. Update INPUT_PATH to the correct location."
    )

df = pd.read_csv(INPUT_PATH)
print(f"Loaded: {INPUT_PATH}")
print("Shape:", df.shape)
display(df.head())

Loaded: ..\Dataset\BlinkIT-Grocery-Data.csv
Shape: (8523, 12)


,Item Fat Content,"""Item Identifier""","""Item Type""","""Outlet Establishment Year""","""Outlet Identifier""","""Outlet Location Type""","""Outlet Size""","""Outlet Type""","""Item Visibility""","""Item Weight""","""Sales""","""Rating"""
0,Regular,"""FDX32""","""Fruits and Vegetables""","""2012""","""OUT049""","""Tier 1""","""Medium""","""Supermarket Type1""","""0.1000135""","""15.1""","""145.4786""","""5"""
1,Low Fat,"""NCB42""","""Health and Hygiene""","""2022""","""OUT018""","""Tier 3""","""Medium""","""Supermarket Type2""","""0.008596051""","""11.8""","""115.3492""","""5"""
2,Regular,"""FDR28""","""Frozen Foods""","""2016""","""OUT046""","""Tier 1""","""Small""","""Supermarket Type1""","""0.025896485""","""13.85""","""165.021""","""5"""
3,Regular,"""FDL50""","""Canned""","""2014""","""OUT013""","""Tier 3""","""High""","""Supermarket Type1""","""0.042277867""","""12.15""","""126.5046""","""5"""
4,Low Fat,"""DRI25""","""Soft Drinks""","""2015""","""OUT045""","""Tier 2""","""Small""","""Supermarket Type1""","""0.033970195""","""19.6""","""55.1614""","""5"""


Display the original dataset dimensions.

In [9]:
df.shape

(8523, 12)

## 3. Capture the original data-quality baseline

Count missing values in the original data.

In [11]:
original_missing = df.isnull().sum()

Calculate original missing-value percentages.

In [12]:
(original_missing / len(df) * 100).round(2)

Item Fat Content               0.000
 "Item Identifier"             0.000
 "Item Type"                   0.000
 "Outlet Establishment Year"   0.000
 "Outlet Identifier"           0.000
 "Outlet Location Type"        0.000
 "Outlet Size"                 0.000
 "Outlet Type"                 0.000
 "Item Visibility"             0.000
 "Item Weight"                 0.000
 "Sales"                       0.000
 "Rating"                      0.000
dtype: float64

Count exact duplicate rows in the original data.

In [97]:
original_duplicate_count = int(df.duplicated().sum())

In [98]:
original_duplicate_count

0

Create a baseline summary.

In [99]:
baseline_summary = pd.DataFrame({
    'Missing Values': original_missing,
    'Missing Percentage': (original_missing / len(df) * 100).round(2)
})

baseline_summary

,Missing Values,Missing Percentage
Item Fat Content,0,0.000
"""Item Identifier""",0,0.000
"""Item Type""",0,0.000
"""Outlet Establishment Year""",0,0.000
"""Outlet Identifier""",0,0.000
"""Outlet Location Type""",0,0.000
"""Outlet Size""",0,0.000
"""Outlet Type""",0,0.000
"""Item Visibility""",0,0.000
"""Item Weight""",0,0.000


## 4. Create a working copy

Copy the original DataFrame so the raw data remains unchanged.

In [100]:
clean_df = df.copy()

Identify text columns.

In [101]:
text_columns = clean_df.select_dtypes(include=['object', 'category']).columns

C:\Users\Sripada Pavankumar\AppData\Local\Temp\ipykernel_7280\166469716.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = clean_df.select_dtypes(include=['object', 'category']).columns


Remove leading and trailing spaces from text columns.

In [102]:
for column in text_columns:
    clean_df[column] = clean_df[column].astype('string').str.strip()

Display the first five rows of the working copy.

In [103]:
clean_df.head()

,Item Fat Content,"""Item Identifier""","""Item Type""","""Outlet Establishment Year""","""Outlet Identifier""","""Outlet Location Type""","""Outlet Size""","""Outlet Type""","""Item Visibility""","""Item Weight""","""Sales""","""Rating"""
0,Regular,"""FDX32""","""Fruits and Vegetables""","""2012""","""OUT049""","""Tier 1""","""Medium""","""Supermarket Type1""","""0.1000135""","""15.1""","""145.4786""","""5"""
1,Low Fat,"""NCB42""","""Health and Hygiene""","""2022""","""OUT018""","""Tier 3""","""Medium""","""Supermarket Type2""","""0.008596051""","""11.8""","""115.3492""","""5"""
2,Regular,"""FDR28""","""Frozen Foods""","""2016""","""OUT046""","""Tier 1""","""Small""","""Supermarket Type1""","""0.025896485""","""13.85""","""165.021""","""5"""
3,Regular,"""FDL50""","""Canned""","""2014""","""OUT013""","""Tier 3""","""High""","""Supermarket Type1""","""0.042277867""","""12.15""","""126.5046""","""5"""
4,Low Fat,"""DRI25""","""Soft Drinks""","""2015""","""OUT045""","""Tier 2""","""Small""","""Supermarket Type1""","""0.033970195""","""19.6""","""55.1614""","""5"""


## 5. Standardize Item Fat Content

Inspect Item Fat Content values before standardization.

In [104]:
clean_df['Item Fat Content'].value_counts(dropna=False)

Item Fat Content
Low Fat    5089
Regular    2889
LF          316
reg         117
low fat     112
Name: count, dtype: Int64

Create a mapping for inconsistent labels.

In [105]:
fat_content_mapping = {
    'LF': 'Low Fat',
    'low fat': 'Low Fat',
    'reg': 'Regular'
}

Apply the standardization mapping.

In [106]:
clean_df['Item Fat Content'] = clean_df['Item Fat Content'].replace(fat_content_mapping)

Inspect Item Fat Content values after standardization.

In [107]:
clean_df['Item Fat Content'].value_counts(dropna=False)

Item Fat Content
Low Fat    5517
Regular    3006
Name: count, dtype: Int64

## 6. Handle missing Item Weight values

Count missing Item Weight values before imputation.

In [108]:
# Normalize headers and values, then convert numeric columns.
clean_df.columns = [str(c).strip().strip('"').strip("'").strip() for c in clean_df.columns]

for column in clean_df.select_dtypes(include=['object', 'string']).columns:
    clean_df[column] = clean_df[column].astype('string').str.strip().str.strip('"').str.strip("'")
    clean_df[column] = clean_df[column].replace('', pd.NA)

numeric_columns = [
    'Outlet Establishment Year', 'Item Visibility', 'Item Weight',
    'Sales', 'Rating'
]
for column in numeric_columns:
    if column in clean_df.columns:
        clean_df[column] = pd.to_numeric(clean_df[column], errors='coerce')

missing_weight_before = int(clean_df['Item Weight'].isna().sum())

In [109]:
missing_weight_before

1463

Calculate the median Item Weight for each Item Identifier.

In [110]:
item_median_weight = clean_df.groupby('Item Identifier')['Item Weight'].transform('median')

Fill missing Item Weight using the same Item Identifier median.

In [111]:
clean_df['Item Weight'] = clean_df['Item Weight'].fillna(item_median_weight)

Count remaining missing Item Weight values after product-level imputation.

In [112]:
missing_after_item_fill = clean_df['Item Weight'].isna().sum()

In [113]:
missing_after_item_fill

np.int64(4)

Calculate the median Item Weight for each Item Type.

In [114]:
type_median_weight = clean_df.groupby('Item Type')['Item Weight'].transform('median')

Fill remaining missing Item Weight using the Item Type median.

In [115]:
clean_df['Item Weight'] = clean_df['Item Weight'].fillna(type_median_weight)

Count remaining missing Item Weight values after category-level imputation.

In [116]:
missing_after_type_fill = clean_df['Item Weight'].isna().sum()

In [117]:
missing_after_type_fill

np.int64(0)

Calculate the overall median Item Weight for the final fallback.

In [118]:
overall_weight_median = clean_df['Item Weight'].median()

In [119]:
overall_weight_median

np.float64(12.65)

Fill any remaining missing Item Weight values using the overall median.

In [120]:
clean_df['Item Weight'] = clean_df['Item Weight'].fillna(overall_weight_median)

Count missing Item Weight values after all imputation stages.

In [121]:
clean_df['Item Weight'].isna().sum()

np.int64(0)

Compare Item Weight before and after imputation.

In [122]:
pd.DataFrame({
    'Before Cleaning': [missing_weight_before],
    'After Cleaning': [clean_df['Item Weight'].isna().sum()]
}, index=['Missing Item Weight Values'])

,Before Cleaning,After Cleaning
Missing Item Weight Values,1463,0


**Why use this method?** A product identifier is the most specific grouping available for a product's weight. Item Type median is the next fallback, and the overall median is used only if a group has no known weights. Imputed values are estimates and should be documented.

## 7. Investigate Item Visibility zeros

Count zero Item Visibility records.

In [123]:
(clean_df['Item Visibility'] == 0).sum()

np.int64(526)

Calculate the percentage of zero Item Visibility records.

In [124]:
((clean_df['Item Visibility'] == 0).mean() * 100).round(2)

np.float64(6.17)

Decision: retain zero visibility values until the business definition confirms whether they represent valid observations.

## 8. Check duplicate records

Count exact duplicate rows before removal.

In [125]:
duplicates_before = clean_df.duplicated().sum()

In [129]:
print(duplicates_before)

0


Remove exact duplicate rows, if present.

In [127]:
clean_df = clean_df.drop_duplicates().copy()

Count exact duplicate rows after removal.

In [128]:
clean_df.duplicated().sum()

np.int64(0)

## 9. Validate numerical values

Display the numerical summary after cleaning.

In [62]:
clean_df.describe()

,Outlet Establishment Year,Item Visibility,Item Weight,Sales,Rating
count,"8,523.000","8,523.000","8,523.000","8,523.000","8,523.000"
mean,"2,016.451",0.066,12.875,140.993,3.966
std,3.189,0.052,4.645,62.275,0.606
min,"2,011.000",0.000,4.555,31.290,1.000
25%,"2,014.000",0.027,8.785,93.826,4.000
50%,"2,016.000",0.054,12.650,143.013,4.000
75%,"2,018.000",0.095,16.850,185.644,4.200
max,"2,022.000",0.328,21.350,266.888,5.000


Count negative Item Weight values.

In [63]:
(clean_df['Item Weight'] < 0).sum()

np.int64(0)

Count negative Item Visibility values.

In [64]:
(clean_df['Item Visibility'] < 0).sum()

np.int64(0)

Count negative Sales values.

In [65]:
(clean_df['Sales'] < 0).sum()

np.int64(0)

Count Rating values outside the 0 to 5 range.

In [66]:
(~clean_df['Rating'].between(0, 5)).sum()

np.int64(0)

## 10. Detect potential outliers using IQR

Identify numerical columns.

In [67]:
numeric_columns = clean_df.select_dtypes(include=np.number).columns

Calculate the first quartile (Q1) for Sales.

In [68]:
sales_q1 = clean_df['Sales'].quantile(0.25)

Calculate the third quartile (Q3) for Sales.

In [69]:
sales_q3 = clean_df['Sales'].quantile(0.75)

Calculate the interquartile range (IQR) for Sales.

In [70]:
sales_iqr = sales_q3 - sales_q1

Calculate the lower IQR boundary for Sales.

In [71]:
sales_lower_bound = sales_q1 - 1.5 * sales_iqr

Calculate the upper IQR boundary for Sales.

In [72]:
sales_upper_bound = sales_q3 + 1.5 * sales_iqr

Identify Sales records outside the IQR boundaries.

In [73]:
sales_outlier_mask = (clean_df['Sales'] < sales_lower_bound) | (clean_df['Sales'] > sales_upper_bound)

Count potential Sales outliers.

In [74]:
sales_outlier_mask.sum()

np.int64(0)

Display a sample of potential Sales outlier records.

In [75]:
clean_df.loc[sales_outlier_mask].head(20)

,Item Fat Content,Item Identifier,Item Type,Outlet Establishment Year,Outlet Identifier,Outlet Location Type,Outlet Size,Outlet Type,Item Visibility,Item Weight,Sales,Rating


Calculate IQR outlier counts for every numerical column.

In [76]:
outlier_rows = []

for column in numeric_columns:
    q1 = clean_df[column].quantile(0.25)
    q3 = clean_df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    mask = (clean_df[column] < lower_bound) | (clean_df[column] > upper_bound)

    outlier_rows.append({
        'Column': column,
        'Q1': q1,
        'Q3': q3,
        'IQR': iqr,
        'Lower Bound': lower_bound,
        'Upper Bound': upper_bound,
        'Potential Outliers': int(mask.sum())
    })

outlier_summary = pd.DataFrame(outlier_rows)

Display the numerical outlier summary.

In [77]:
outlier_summary

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Potential Outliers
0,Outlet Establishment Year,"2,014.000","2,018.000",4.000,"2,008.000","2,024.000",0
1,Item Visibility,0.027,0.095,0.068,-0.074,0.196,144
2,Item Weight,8.785,16.850,8.065,-3.313,28.948,0
3,Sales,93.826,185.644,91.817,-43.899,323.370,0
4,Rating,4.000,4.200,0.200,3.700,4.500,1931


**Important:** IQR flags statistical extremes, not necessarily errors. Do not remove or cap outliers without validating them against business context.

## 11. Compare before and after cleaning

Count missing cells in the cleaned dataset.

In [78]:
cleaned_missing_count = int(clean_df.isnull().sum().sum())

Count exact duplicates in the cleaned dataset.

In [79]:
cleaned_duplicate_count = int(clean_df.duplicated().sum())

Create a before-and-after comparison table.

In [80]:
comparison = pd.DataFrame({
    'Before Cleaning': [
        len(df),
        df.shape[1],
        int(df.isnull().sum().sum()),
        original_duplicate_count
    ],
    'After Cleaning': [
        len(clean_df),
        clean_df.shape[1],
        cleaned_missing_count,
        cleaned_duplicate_count
    ]
}, index=['Rows', 'Columns', 'Missing Cells', 'Exact Duplicate Rows'])

comparison

,Before Cleaning,After Cleaning
Rows,8523,8523
Columns,12,12
Missing Cells,0,0
Exact Duplicate Rows,0,0


Display missing values by column after cleaning.

In [81]:
clean_df.isnull().sum()

Item Fat Content             0
Item Identifier              0
Item Type                    0
Outlet Establishment Year    0
Outlet Identifier            0
Outlet Location Type         0
Outlet Size                  0
Outlet Type                  0
Item Visibility              0
Item Weight                  0
Sales                        0
Rating                       0
dtype: int64

## 12. Final validation

Confirm that Item Weight has no remaining missing values.

In [82]:
assert clean_df['Item Weight'].isna().sum() == 0

Confirm that Item Fat Content contains only the expected standardized labels.

In [83]:
clean_df['Item Fat Content'].value_counts(dropna=False)

Item Fat Content
Low Fat    5517
Regular    3006
Name: count, dtype: Int64

Confirm that no exact duplicate rows remain.

In [84]:
assert clean_df.duplicated().sum() == 0

Display the final cleaned dataset dimensions.

In [85]:
clean_df.shape

(8523, 12)

Display the first five cleaned records.

In [86]:
clean_df.head()

,Item Fat Content,Item Identifier,Item Type,Outlet Establishment Year,Outlet Identifier,Outlet Location Type,Outlet Size,Outlet Type,Item Visibility,Item Weight,Sales,Rating
0,Regular,FDX32,Fruits and Vegetables,2012,OUT049,Tier 1,Medium,Supermarket Type1,0.100,15.100,145.479,5.000
1,Low Fat,NCB42,Health and Hygiene,2022,OUT018,Tier 3,Medium,Supermarket Type2,0.009,11.800,115.349,5.000
2,Regular,FDR28,Frozen Foods,2016,OUT046,Tier 1,Small,Supermarket Type1,0.026,13.850,165.021,5.000
3,Regular,FDL50,Canned,2014,OUT013,Tier 3,High,Supermarket Type1,0.042,12.150,126.505,5.000
4,Low Fat,DRI25,Soft Drinks,2015,OUT045,Tier 2,Small,Supermarket Type1,0.034,19.600,55.161,5.000


## 13. Save the cleaned dataset

Set the output path inside the Dataset folder.

In [ ]:
OUTPUT_PATH = Path('../Dataset/blinkit_sales_cleaned.csv')

Create the Dataset folder if it does not exist.

In [88]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

Save the cleaned DataFrame as a new CSV without the DataFrame index.

In [89]:
clean_df.to_csv(OUTPUT_PATH, index=False)

Display the full output path.

In [90]:
print(f'Cleaned dataset saved to: {OUTPUT_PATH.resolve()}')

Cleaned dataset saved to: C:\Users\Sripada Pavankumar\Desktop\pa1\blinkit_data_analysis\dataset\blinkit_sales_cleaned_123.csv


## 14. Cleaning summary

Create a final cleaning summary.

In [91]:
cleaning_summary = pd.DataFrame({
    'Metric': [
        'Original rows',
        'Cleaned rows',
        'Original missing cells',
        'Remaining missing cells',
        'Original duplicate rows',
        'Remaining duplicate rows',
        'Zero Item Visibility records',
        'Potential Sales outliers (IQR)'
    ],
    'Value': [
        len(df),
        len(clean_df),
        int(df.isnull().sum().sum()),
        int(clean_df.isnull().sum().sum()),
        original_duplicate_count,
        int(clean_df.duplicated().sum()),
        int((clean_df['Item Visibility'] == 0).sum()),
        int(sales_outlier_mask.sum())
    ]
})

Display the cleaning summary.

In [92]:
cleaning_summary

,Metric,Value
0,Original rows,8523
1,Cleaned rows,8523
2,Original missing cells,0
3,Remaining missing cells,0
4,Original duplicate rows,0
5,Remaining duplicate rows,0
6,Zero Item Visibility records,526
7,Potential Sales outliers (IQR),0


## 15. Analyst notes and limitations

Record the decisions and observations from your run:

- **Item Fat Content:** inconsistent labels standardized.
- **Item Weight:** missing values imputed using product, category, and overall medians in sequence.
- **Item Visibility:** zero values retained pending business validation.
- **Duplicates:** exact duplicates removed if found.
- **Outliers:** detected using IQR; not automatically removed.
- **Source data:** original CSV is preserved.
- **Output:** `Dataset/blinkit_sales_cleaned.csv`.